# Regular TensorFlow vs TinyML Audio Classification

This notebook compares a regular TensorFlow/Keras audio classifier with a quantized TensorFlow Lite model for a small sound-classification task. The regular model is a laptop baseline. The TinyML/TFLite model is the candidate style for the later ESP32 + INMP441 audio node.

The comparison does **not** require the physical microphone yet. It uses a reproducible audio dataset on the laptop or in Colab. Later, the same output idea can be connected to openHAB through MQTT with `label`, `confidence`, `rms`, and `inference_ms` instead of raw audio.

Recommended kernels:

- Local VS Code kernel: `Safety TinyML (.venv)` for editing and light runs.
- Colab runtime/kernel: heavier TensorFlow training if the local machine is slow.

## Comparison Idea

- **Regular model:** trained and evaluated as a normal Keras model on the laptop.
- **Tiny model:** converted from the same model to TensorFlow Lite with int8 quantization.
- **Input:** one-second audio windows transformed into spectrograms.
- **Output classes:** `yes`, `no`, `unknown`, `silence`.

The `yes/no/unknown/silence` classes are not the final safety labels. They are a clean first experiment because the dataset is available and reproducible. In the report, this can be explained as a proxy task that demonstrates the technical difference between regular inference and TinyML inference. The later safety node can use labels such as `normal`, `knock`, `alarm_like`, and `silence` once real data is collected.

In [ ]:
# Run this first. A Colab kernel is remote and does not use the local .venv.
# Therefore Colab installs packages here, while the local VS Code kernel uses requirements.txt.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Colab runtime detected. Installing notebook dependencies in the remote Colab environment...')
    get_ipython().run_line_magic('pip', 'install -q tensorflow numpy matplotlib scikit-learn')
else:
    print('Local kernel detected. Use the local .venv and install dependencies with:')
    print('python -m pip install -r requirements.txt')

In [ ]:
from pathlib import Path
import os
import time
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print('TensorFlow:', tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

cwd = Path.cwd()
if cwd.name == 'notebooks' and cwd.parent.name == 'tinyml':
    TINYML_ROOT = cwd.parent
elif (cwd / 'tinyml').exists():
    TINYML_ROOT = cwd / 'tinyml'
else:
    TINYML_ROOT = cwd

print('TinyML root:', TINYML_ROOT)

## Dataset

This uses Google's mini Speech Commands dataset. It is small enough for a laptop experiment and stable enough for a course comparison. The `unknown` class is made from commands that are not `yes` or `no`; the `silence` class is generated as low-amplitude noise.

In [2]:
DATA_URL = 'http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip'
DATA_ROOT = (TINYML_ROOT / 'data' / 'raw').resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_path = tf.keras.utils.get_file(
    origin=DATA_URL,
    fname='mini_speech_commands.zip',
    cache_dir=str(DATA_ROOT),
    cache_subdir='.',
    extract=True,
)

dataset_dir = Path(zip_path).with_suffix('')
print(dataset_dir)
print(sorted([p.name for p in dataset_dir.iterdir() if p.is_dir()]))

182082353/182082353 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
/content/data/raw/mini_speech_commands_extracted
['__MACOSX', 'mini_speech_commands']


In [3]:
TARGET_WORDS = ['yes', 'no']
ALL_WORD_DIRS = sorted([p for p in dataset_dir.iterdir() if p.is_dir()])
UNKNOWN_WORDS = [p.name for p in ALL_WORD_DIRS if p.name not in TARGET_WORDS]
LABELS = ['yes', 'no', 'unknown', 'silence']
label_to_index = {name: i for i, name in enumerate(LABELS)}

MAX_PER_CLASS = 800

def collect_files():
    paths = []
    labels = []

    for word in TARGET_WORDS:
        files = sorted((dataset_dir / word).glob('*.wav'))[:MAX_PER_CLASS]
        paths.extend(files)
        labels.extend([label_to_index[word]] * len(files))

    unknown_files = []
    per_unknown_word = max(1, MAX_PER_CLASS // len(UNKNOWN_WORDS))
    for word in UNKNOWN_WORDS:
        unknown_files.extend(sorted((dataset_dir / word).glob('*.wav'))[:per_unknown_word])
    unknown_files = unknown_files[:MAX_PER_CLASS]
    paths.extend(unknown_files)
    labels.extend([label_to_index['unknown']] * len(unknown_files))

    return np.array([str(p) for p in paths]), np.array(labels, dtype=np.int64)

paths, labels = collect_files()
print('Audio files:', len(paths))
for label_name in LABELS[:-1]:
    print(label_name, np.sum(labels == label_to_index[label_name]))

Audio files: 0
yes 0
no 0
unknown 0


## Audio Preprocessing

The ESP32 would later receive raw samples from the INMP441 microphone. For this notebook, the laptop reads `.wav` files. Both paths can use the same conceptual preprocessing step: convert a short waveform window into a compact feature representation. Here we use a spectrogram.

In [4]:
SAMPLE_RATE = 16000
WINDOW_SAMPLES = 16000

def decode_audio(path):
    audio_binary = tf.io.read_file(path)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = audio[:WINDOW_SAMPLES]
    padding = WINDOW_SAMPLES - tf.shape(audio)[0]
    audio = tf.pad(audio, [[0, padding]])
    return audio

def waveform_to_spectrogram(waveform):
    spectrogram = tf.signal.stft(waveform, frame_length=255, frame_step=128)
    spectrogram = tf.abs(spectrogram)
    spectrogram = spectrogram[..., tf.newaxis]
    return spectrogram

def file_to_example(path, label):
    waveform = decode_audio(path)
    return waveform_to_spectrogram(waveform), label

def make_silence(count):
    waveforms = tf.random.normal([count, WINDOW_SAMPLES], mean=0.0, stddev=0.005, seed=42)
    spectrograms = tf.map_fn(waveform_to_spectrogram, waveforms, fn_output_signature=tf.float32)
    labels = tf.fill([count], label_to_index['silence'])
    return spectrograms, labels

In [5]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(paths))
paths = paths[indices]
labels = labels[indices]

n_train = int(0.7 * len(paths))
n_val = int(0.15 * len(paths))

train_paths, train_labels = paths[:n_train], labels[:n_train]
val_paths, val_labels = paths[n_train:n_train+n_val], labels[n_train:n_train+n_val]
test_paths, test_labels = paths[n_train+n_val:], labels[n_train+n_val:]

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def make_file_dataset(ds_paths, ds_labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((ds_paths, ds_labels))
    if shuffle:
        ds = ds.shuffle(len(ds_paths), seed=42)
    return ds.map(file_to_example, num_parallel_calls=AUTOTUNE)

train_ds = make_file_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_file_dataset(val_paths, val_labels)
test_ds = make_file_dataset(test_paths, test_labels)

sil_train_x, sil_train_y = make_silence(MAX_PER_CLASS)
sil_val_x, sil_val_y = make_silence(120)
sil_test_x, sil_test_y = make_silence(120)

train_ds = train_ds.concatenate(tf.data.Dataset.from_tensor_slices((sil_train_x, sil_train_y)))
val_ds = val_ds.concatenate(tf.data.Dataset.from_tensor_slices((sil_val_x, sil_val_y)))
test_ds = test_ds.concatenate(tf.data.Dataset.from_tensor_slices((sil_test_x, sil_test_y)))

train_ds = train_ds.shuffle(4000, seed=42).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

for example_specs, example_labels in train_ds.take(1):
    INPUT_SHAPE = example_specs.shape[1:]
    print('Input shape:', INPUT_SHAPE)
    print('Labels:', example_labels[:8].numpy())

InvalidArgumentError: {{function_node __wrapped__ShuffleDatasetV3_device_/job:localhost/replica:0/task:0/device:CPU:0}} buffer_size must be greater than zero or UNKNOWN_CARDINALITY [Op:ShuffleDatasetV3] name: 

In [ ]:
plt.figure(figsize=(8, 4))
for specs, batch_labels in train_ds.take(1):
    for i in range(4):
        plt.subplot(1, 4, i + 1)
        plt.imshow(tf.squeeze(specs[i]).numpy().T, aspect='auto', origin='lower')
        plt.title(LABELS[int(batch_labels[i])])
        plt.axis('off')
plt.tight_layout()

## Regular TensorFlow Model

This is the laptop baseline. It is not required to fit on the ESP32. Its role is to show what a normal TensorFlow workflow looks like when memory and deployment constraints are less strict.

In [ ]:
regular_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    tf.keras.layers.Rescaling(1.0 / 255.0),
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(LABELS), activation='softmax'),
])

regular_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

regular_model.summary()

In [ ]:
history = regular_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)

In [ ]:
regular_loss, regular_accuracy = regular_model.evaluate(test_ds)
print('Regular test accuracy:', regular_accuracy)

## Convert To TensorFlow Lite And Quantize

The TinyML version uses TensorFlow Lite conversion and int8 quantization. This is the relevant direction for microcontrollers because it reduces model size and uses integer arithmetic. The notebook evaluates it on the laptop, but the artifact is closer to what the ESP32 would use through TFLite Micro.

In [ ]:
MODELS_DIR = (TINYML_ROOT / 'models').resolve()
EXPORTED_DIR = (TINYML_ROOT / 'exported').resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPORTED_DIR.mkdir(parents=True, exist_ok=True)

regular_keras_path = MODELS_DIR / 'regular_model.keras'
regular_tflite_path = MODELS_DIR / 'regular_model_float32.tflite'
tiny_tflite_path = MODELS_DIR / 'tiny_model_int8.tflite'

regular_model.save(regular_keras_path)

converter = tf.lite.TFLiteConverter.from_keras_model(regular_model)
regular_tflite = converter.convert()
regular_tflite_path.write_bytes(regular_tflite)

def representative_dataset():
    for specs, _ in train_ds.unbatch().batch(1).take(100):
        yield [tf.cast(specs, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(regular_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tiny_tflite = converter.convert()
tiny_tflite_path.write_bytes(tiny_tflite)

print('Saved:', regular_keras_path)
print('Saved:', regular_tflite_path)
print('Saved:', tiny_tflite_path)

In [ ]:
def evaluate_tflite_model(tflite_path, dataset):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = input_details['quantization']
    output_scale, output_zero_point = output_details['quantization']

    correct = 0
    total = 0
    y_true = []
    y_pred = []
    timings_ms = []

    for specs, labels_batch in dataset.unbatch().batch(1):
        input_data = specs.numpy().astype(np.float32)
        if input_details['dtype'] == np.int8:
            input_data = input_data / input_scale + input_zero_point
            input_data = np.clip(input_data, -128, 127).astype(np.int8)

        start = time.perf_counter()
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        elapsed_ms = (time.perf_counter() - start) * 1000

        output = interpreter.get_tensor(output_details['index'])[0]
        if output_details['dtype'] == np.int8:
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        pred = int(np.argmax(output))
        true = int(labels_batch.numpy()[0])
        correct += int(pred == true)
        total += 1
        y_true.append(true)
        y_pred.append(pred)
        timings_ms.append(elapsed_ms)

    return {
        'accuracy': correct / total,
        'avg_inference_ms_laptop': float(np.mean(timings_ms)),
        'y_true': y_true,
        'y_pred': y_pred,
    }

regular_tflite_eval = evaluate_tflite_model(regular_tflite_path, test_ds)
tiny_tflite_eval = evaluate_tflite_model(tiny_tflite_path, test_ds)

regular_tflite_eval['accuracy'], tiny_tflite_eval['accuracy']

In [ ]:
size_regular_keras = regular_keras_path.stat().st_size
size_regular_tflite = regular_tflite_path.stat().st_size
size_tiny_tflite = tiny_tflite_path.stat().st_size

comparison = {
    'metric': ['target', 'numeric_format', 'model_size_bytes', 'accuracy', 'avg_inference_ms_laptop', 'openhab_payload'],
    'regular_tensorflow': [
        'Laptop baseline',
        'float32',
        size_regular_keras,
        float(regular_accuracy),
        'not measured here',
        'could send raw/central features, but not privacy-friendly',
    ],
    'tinyml_tflite_int8': [
        'ESP32 candidate style',
        'int8',
        size_tiny_tflite,
        tiny_tflite_eval['accuracy'],
        tiny_tflite_eval['avg_inference_ms_laptop'],
        'label/confidence/rms/inference_ms only',
    ],
}

for row in zip(comparison['metric'], comparison['regular_tensorflow'], comparison['tinyml_tflite_int8']):
    print(f'{row[0]:28s} | {str(row[1]):35s} | {row[2]}')

In [ ]:
cm = confusion_matrix(tiny_tflite_eval['y_true'], tiny_tflite_eval['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('TinyML/TFLite int8 confusion matrix')
plt.tight_layout()

## Export Tiny Model As C Array

TFLite Micro examples often include the model as a C/C++ byte array. This export is not the full ESP32 firmware yet, but it creates the artifact shape that firmware would include later.

In [ ]:
def bytes_to_c_array(data, variable_name):
    hex_values = ', '.join(f'0x{byte:02x}' for byte in data)
    return (
        f'const unsigned char {variable_name}[] = {{{hex_values}}};\n'
        f'const unsigned int {variable_name}_len = {len(data)};\n'
    )

c_path = EXPORTED_DIR / 'tiny_model_int8.cc'
c_path.write_text(bytes_to_c_array(tiny_tflite_path.read_bytes(), 'tiny_model_int8'), encoding='utf-8')
print('Saved:', c_path)
print('C array size:', c_path.stat().st_size, 'bytes')

## Connection To The openHAB Safety System

The deployed audio node should not send continuous raw audio to openHAB. It should publish summarized inference data:

```json
{
  "label": "alarm_like",
  "confidence": 0.87,
  "rms": 421,
  "inference_ms": 18,
  "model": "sound_v1_int8"
}
```

Planned MQTT topic:

```text
tele/safety_audio_1/CLASSIFICATION
```

Raw samples are still inspectable during development through the notebook, serial monitor, or temporary debug MQTT topics. They are not part of the normal openHAB data path.

## Interpretation Template

Use the results above to fill this in:

- The regular TensorFlow model is easier to train and inspect on the laptop, but it is not the target deployment format for the ESP32.
- The quantized TFLite model is smaller and closer to TinyML deployment, but may lose some accuracy.
- Both models solve the same classification task and produce the same label set.
- The TinyML version changes the system architecture: inference moves to the edge node, and openHAB receives only summarized data.
- This supports the safety-monitoring concept because sound context can influence the risk score without streaming raw microphone data.